## Задание 1, 2 и 3

В этой серии заданий вам понадобится сделать эмбеддинги для объектов из некоторой подвыборки датасета IMDB с помощью предобученных моделей из Hugging Face.

В этом задании сделайте их с помощью модели BERT (bert-base-cased), RoBERTa (roberta-base), DistilBERT (distilbert-base-cased).



In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from tqdm import tqdm
from transformers import AutoTokenizer
from transformers import BertModel
from transformers import DataCollatorWithPadding
from transformers import DistilBertModel
from transformers import RobertaModel


def get_model(model_name):
    assert model_name in ['bert', 'roberta', 'distilbert']

    checkpoint_names = {
        'bert': 'bert-base-cased',
        'roberta': 'roberta-base',
        'distilbert': 'distilbert-base-cased'
    }

    model_classes = {
        'bert': BertModel,
        'roberta': RobertaModel,
        'distilbert': DistilBertModel
    }

    return (
        AutoTokenizer.from_pretrained(checkpoint_names[model_name]),
        model_classes[model_name].from_pretrained(checkpoint_names[model_name])
    )


@torch.inference_mode()
def get_embeddings(model, loader, device):
    model.eval()

    embeddings = []

    for batch in tqdm(loader):
        batch = {key: batch[key].to(device) for key in ['attention_mask', 'input_ids']}

        embeddings.append(model(**batch)['last_hidden_state'][:, 0, :].cpu())

    return torch.cat(embeddings, dim=0)


def main(model_name: str):
    tokenizer, model = get_model(model_name)

    dataset = load_dataset("imdb", split="train")

    def tokenization(example):
        return tokenizer.batch_encode_plus(
            example['text'],
            add_special_tokens=True,
            return_token_type_ids=False,
            truncation=True
        )

    dataset = dataset.map(tokenization, batched=True)
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    
    np.random.seed(100)
    idx = np.random.randint(len(dataset), size=200).tolist()

    loader = DataLoader(
        Subset(dataset, idx),
        batch_size=16,
        collate_fn=DataCollatorWithPadding(tokenizer=tokenizer),
        pin_memory=True,
        shuffle=False
    )

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    model = model.to(device)
    
    embeddings = get_embeddings(model, loader, device)

    torch.save(embeddings, f'embeddings_{model_name}.pt')

In [ ]:
main('bert')

In [ ]:
main('roberta')

In [ ]:
main('distilbert')